In [ ]:
!pip install streamlit pyngrok python-docx transformers accelerate bitsandbytes pandas numpy
!pip uninstall pyarrow datasets -y
!pip install datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 84.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 28.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 108.4 MB/s eta 0:00:00
Found existing installation: pyarrow 18.1.0
Uninstalling pyarrow-18.1.0:
  Successfully uninstalled pyarrow-18.1.0
Found existing installation: datasets 4.0.0
Uninstalling datasets-4.0.0:
  Successfully uninstalled datasets-4.0.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 15.1 MB/s eta 0:00:00


In [ ]:
import os
import time
import torch
import gc

os.system('pkill -f streamlit')
os.system('pkill -f ngrok')

if os.path.exists('app.py'):
    os.remove('app.py')

print("Готово к созданию app.py")

Готово к созданию app.py


In [ ]:
app_code = '''import streamlit as st
import time
import torch
import gc
import re
from transformers import AutoTokenizer, BitsAndBytesConfig, AutoModelForCausalLM

st.set_page_config(
    page_title="Автоматическое кодирование интервью",
    layout="wide",
    initial_sidebar_state="expanded"
)

st.title("Автоматическое кодирование интервью")
st.markdown("### Генерация тематических кодов и цитат из текстов интервью")
st.markdown("---")

if 'model_loaded' not in st.session_state:
    st.session_state.model_loaded = False

with st.sidebar:
    st.header("Настройки")
    max_new_tokens = st.slider("Макс. токенов", 512, 4096, 2048, 128)
    num_beams = st.slider("Beam search", 1, 5, 2, 1)
    temperature = st.slider("Температура", 0.0, 1.0, 0.0, 0.1)
    st.markdown("---")
    st.info("Приложение кодирует интервью с помощью Qwen2.5-7B-Instruct")

@st.cache_resource
def load_model():
    with st.spinner("Загрузка модели..."):
        base_model = "Qwen/Qwen2.5-7B-Instruct"
        tokenizer = AutoTokenizer.from_pretrained(base_model, trust_remote_code=True)
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=torch.bfloat16
        )
        model = AutoModelForCausalLM.from_pretrained(
            base_model,
            quantization_config=bnb_config,
            device_map="auto",
            torch_dtype=torch.bfloat16
        )
        model.eval()
        st.success("Модель загружена!")
        return model, tokenizer

def build_prompt_h4(example, tokenizer):
    instruction = """You are an expert in interview analysis. Your task is to highlight the thematic codes in the interview text and corresponding quotes.

Act step by step:

1. Carefully review the output format shown in the example below. Remember that each code block must contain "Общий код" (General code), then Quote, then "Конкретный код" (Specific code). The quote must be verbatim and enclosed in quotation marks.
2. Read the interview transcript and the topic. Identify all fragments (quotes) that relate to the interview topic.
3. Group the quotes by general themes — these will be the "Общий код" (General codes). For each general theme, come up with a short name.
4. Within each general code, identify specific meaning aspects — these will be the "конкретный код" (Specific codes). The names of specific codes should reflect the essence of the quote.
5. Generate the answer strictly following the format from the example. Do not add any explanations, do not write words like 'Step 1', 'Step 2' — only the final blocks of codes and quotes.

Format of an output (consists of several general codes, each followed by quotes and specific codes):
**Общий код 1: <generate general code 1>**
"<quote text>" - **<generate specific code 1> (Конкретный код)**
"<quote text>" - **<generate specific code 2> (Конкретный код)**
**Общий код 2: <generate general code 2>**
"<quote text>" - **<generate specific code> (Конкретный код)**
**Общий код <general code number>: <generate general code>**
"<quote text>" - **<generate specific code> (Конкретный код)**
and so on. You choose the number of general and specific codes.

Example of a topic:
ПОКОЛЕНЧЕСКАЯ ИДЕНТИЧНОСТЬ, ЖИЗНЕННЫЕ ВЫБОРЫ И УСТОЙЧИВОСТЬ РОССИЙСКОЙ МОЛОДЕЖИ

Example of a general code related to the topic (pay attention to the structure):
**Общий код 1: Поколенческие характеристики, ценности и жизненные ориентиры**
"Мне кажется, большинство особо не стремятся там вот срочно, прямо сейчас там жениться, замуж, там детей и так далее. Вот. То есть как-то больше сосредоточены даже не на карьере, а на себе, на том, чтобы сложить все для себя вот так, как хочется, да. То есть не просто чтобы там выйти замуж, а чтобы выйти замуж вот по любви, чтобы все было идеально. Вот на какой-то такой идеальности что ли." - **Фокус на самореализации и качественных отношениях (конкретный код)**
"У нашего поколения все-таки все по-другому. Нам не дадут квартиру просто так. Нам не обязательно так просто получить место там где-то на работе и так далее, да. Но при этом у нас гораздо больше возможностей в плане, как сказать, чему-то научиться новому, куда-то поехать, что-то посмотреть, составить свое мнение, там высказать свое мнение даже." - **Осознание свободы выбора и новых возможностей (конкретный код)**
"У детей нынешних у них как будто меньше табу что ли. [...] они спокойно со мной могла поговорить на какие-то откровенные темы, которые мне в ее возрасте, я тоже задумывалась об этом, у меня тоже было какое-то мнение, но я боялась об этом говорить со взрослыми, потому что это было табуировано. [...] Поэтому у них мне кажется растет какое-то более свободное поколение что ли." - **Сравнение с младшим поколением: свобода от табу (конкретный код)**
"Мне кажется, что поколение у нас достаточно трудолюбивое при этом как бы. То есть если человек чего-то хочет добиться в карьерной сфере, ну, человек действительно может приложить там все усилия и добиться этого. Вот. Потому что возможностей супер много сейчас." - **Трудолюбие и вера в возможности (конкретный код)**

Now you should do the markup for the interview according to the plan. Important: The answer should contain only codes and quotes, without unnecessary words and repetitions.
The quotes must be strictly from the text. Give the answer in Russian."""

    user_content = f"""Тема интервью:
{example['topic']}

Текст интервью:
{example['transcript']}

Пожалуйста, выполни разметку интервью, каждый общий код в указанном формате:
**Общий код <generate general code>: <general code name>**
"<quote text>" - **<generate specific code> (Конкретный код)**"""

    messages = [
        {"role": "system", "content": instruction},
        {"role": "user", "content": user_content},
    ]
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    return prompt

def generate(model, tokenizer, prompt, max_tokens, beams, temp):
    torch.cuda.empty_cache()
    gc.collect()
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=4096)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        if temp > 0:
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_tokens,
                do_sample=True,
                temperature=temp,
                top_p=0.95,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id
            )
        else:
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_tokens,
                do_sample=False,
                num_beams=beams,
                early_stopping=True,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id
            )

    result = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    del inputs, outputs
    torch.cuda.empty_cache()
    gc.collect()
    return result

def format_codes_output(text):
    if not text:
        return text

    # Добавляем перенос после каждого "**Общий код"
    text = re.sub(r'(\*\*Общий код \d+:)', r'\\n\\n\\1', text)

    # Добавляем перенос после каждой цитаты перед следующим кодом
    text = re.sub(r'(Конкретный код)\\*\\*', r'\\1**\\n', text)

    # Разделяем блоки двойным переносом
    text = re.sub(r'\\n\\*\\*Общий код', r'\\n\\n**Общий код', text)

    # Убираем лишние переносы
    text = text.strip()

    return text

def main():
    if not st.session_state.model_loaded:
        try:
            model, tokenizer = load_model()
            st.session_state.model = model
            st.session_state.tokenizer = tokenizer
            st.session_state.model_loaded = True
        except Exception as e:
            st.error(f"Ошибка: {e}")
            st.stop()
    else:
        model = st.session_state.model
        tokenizer = st.session_state.tokenizer

    st.header("Кодирование интервью")
    st.markdown("---")

    with st.expander("Пример формата вывода", expanded=False):
        st.markdown("""
        **Общий код 1: Поколенческие характеристики, ценности и жизненные ориентиры**

        "Мне кажется, большинство не стремятся срочно жениться..." - **Фокус на самореализации (Конкретный код)**

        **Общий код 2: Ценности и ориентиры**

        "Человек может приложить усилия и добиться..." - **Трудолюбие и вера в возможности (Конкретный код)**
        """)

    with st.form("main_form"):
        topic = st.text_area("**Тема интервью**", height=100, placeholder="Введите тему интервью...")
        transcript = st.text_area("**Текст интервью**", height=400, placeholder="Введите текст интервью для анализа...")
        if transcript:
            st.caption(f"Длина текста: {len(transcript)} символов")
        submitted = st.form_submit_button("Выполнить кодирование", type="primary")

    if submitted:
        if not topic or not transcript:
            st.error("Заполните все поля!")
        else:
            with st.spinner("Генерация кодов и цитат..."):
                example = {'topic': topic, 'transcript': transcript}
                prompt = build_prompt_h4(example, tokenizer)
                result = generate(model, tokenizer, prompt, max_new_tokens, num_beams, temperature)

                st.success("Кодирование завершено!")
                st.markdown("### Результат")

                # Применяем форматирование
                formatted_result = format_codes_output(result)
                st.markdown(formatted_result)

                # Кнопка скачивания
                st.download_button("Скачать TXT", result, f"coding_{int(time.time())}.txt")

if __name__ == "__main__":
    main()
'''

with open('app.py', 'w', encoding='utf-8') as f:
    f.write(app_code)


<>:146: SyntaxWarning: invalid escape sequence '\*'
<>:146: SyntaxWarning: invalid escape sequence '\*'
/tmp/ipykernel_565/3196908351.py:146: SyntaxWarning: invalid escape sequence '\*'
  text = re.sub(r'(\*\*Общий код \d+:)', r'\\n\\n\\1', text)


In [ ]:
import os
import time
import torch
import gc

os.system('pkill -f streamlit')
os.system('pkill -f ngrok')
time.sleep(2)

if torch.cuda.is_available():
    torch.cuda.empty_cache()
gc.collect()

os.system('streamlit run app.py --server.port 8501 --server.address 0.0.0.0 --server.headless true &')
time.sleep(5)

from pyngrok import ngrok
ngrok.kill()
ngrok.set_auth_token("3DwzqhWEX9aUEF5spTUYpBMceBa_7b6JgtJfqNc9JCqZeX3Vd")
public_url = ngrok.connect(8501)

print(f"\nПубличный URL: {public_url}")
print("Откройте ссылку в браузере")


Публичный URL: NgrokTunnel: "https://cassette-sequester-video.ngrok-free.dev" -> "http://localhost:8501"
Откройте ссылку в браузере
